In [ ]:
# ==========================================
# 0. IMPORT THƯ VIỆN CẦN THIẾT
# ==========================================
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns
from torchsummary import summary

# Cấu hình thiết bị
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
plt.style.use('seaborn-v0_8-muted')
print(f"Đang sử dụng thiết bị: {device}")


In [ ]:
# =======================
# 1. TIỀN XỬ LÝ DỮ LIỆU
# =======================

# Đọc dữ liệu
df = pd.read_csv('/kaggle/input/datasets/gova26/airpressure/Folds5x2_pp.csv')

# Xóa khoảng trắng và ký tự lạ ở tên cột
df.columns = df.columns.str.replace('#', '').str.strip()

# Danh sách cột đầu vào và mục tiêu
features = ['AT', 'V', 'AP', 'RH']
target = 'PE'

# Ép kiểu dữ liệu sang số
for col in features + [target]:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Xóa các hàng chứa giá trị NaN
df.dropna(inplace=True)

print("Kiểu dữ liệu:\n", df.dtypes)

# =======================
# XỬ LÝ OUTLIER
# =======================

Q1 = df[features].quantile(0.25)
Q3 = df[features].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

df_clean = df[
    ~((df[features] < lower_bound) |
      (df[features] > upper_bound)).any(axis=1)
].copy()

print(f"\nSố lượng mẫu sau xử lý: {len(df_clean)}")

# =======================
# CHUẨN BỊ DỮ LIỆU
# =======================

X = df_clean[features].values
y = df_clean[target].values.reshape(-1, 1)

# =======================
# CHIA DỮ LIỆU 70:10:20
# =======================

# Bước 1: Tách 20% dữ liệu làm tập Test
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

# Bước 2: Tách 12.5% của 80% còn lại làm Validation
# 12.5% x 80% = 10% tổng dữ liệu
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size=0.125,
    random_state=42
)

print("\nKích thước các tập dữ liệu:")
print(f"Training:   {len(X_train)} mẫu ({len(X_train)/len(X)*100:.1f}%)")
print(f"Validation: {len(X_val)} mẫu ({len(X_val)/len(X)*100:.1f}%)")
print(f"Testing:    {len(X_test)} mẫu ({len(X_test)/len(X)*100:.1f}%)")

# =======================
# CHUẨN HÓA DỮ LIỆU
# =======================

# Chỉ FIT trên tập Training
scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_train_scaled = scaler_X.fit_transform(X_train)
X_val_scaled = scaler_X.transform(X_val)
X_test_scaled = scaler_X.transform(X_test)

y_train_scaled = scaler_y.fit_transform(y_train)
y_val_scaled = scaler_y.transform(y_val)
y_test_scaled = scaler_y.transform(y_test)

# =======================
# TẠO TENSOR DATASET
# =======================

X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train_scaled, dtype=torch.float32)

X_val_tensor = torch.tensor(X_val_scaled, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val_scaled, dtype=torch.float32)

X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test_scaled, dtype=torch.float32)

# =======================
# TẠO DATALOADER
# =======================

train_loader = DataLoader(
    TensorDataset(X_train_tensor, y_train_tensor),
    batch_size=128,
    shuffle=True
)

val_loader = DataLoader(
    TensorDataset(X_val_tensor, y_val_tensor),
    batch_size=128,
    shuffle=False
)

test_loader = DataLoader(
    TensorDataset(X_test_tensor, y_test_tensor),
    batch_size=128,
    shuffle=False
)


In [ ]:
# ==========================================
# 2. XÂY DỰNG MÔ HÌNH ANN
# ==========================================

class CCPP_Forecaster(nn.Module):
    def __init__(self, input_size):
        super(CCPP_Forecaster, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )
        
    def forward(self, x):
        return self.net(x)        
model = CCPP_Forecaster(len(features)).to(device)
print("\n--- TÓM TẮT KIẾN TRÚC MÔ HÌNH ---")
summary(model, (len(features),))


In [ ]:
# ==========================================
# 3. HUẤN LUYỆN MÔ HÌNH
# ==========================================

# Hàm mất mát và bộ tối ưu
criterion = nn.HuberLoss(delta=1.0)
optimizer = optim.Adam(model.parameters(), lr=0.005)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=15
)

epochs = 200
train_losses, val_losses = [], []
best_val_loss = float('inf')

print("\n--- QUÁ TRÌNH HUẤN LUYỆN ---")

for epoch in range(epochs):

    # ======================================
    # 1. TRAINING
    # ======================================
    model.train()
    r_train_loss = 0

    for inputs, labels in train_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(inputs)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        r_train_loss += loss.item() * inputs.size(0)

    # ======================================
    # 2. VALIDATION
    # ======================================
    model.eval()
    r_val_loss = 0

    with torch.no_grad():
        for inputs, labels in val_loader:  # SỬ DỤNG val_loader
            inputs = inputs.to(device)
            labels = labels.to(device)

            outputs = model(inputs)
            loss = criterion(outputs, labels)

            r_val_loss += loss.item() * inputs.size(0)

    # ======================================
    # 3. TÍNH TRAIN LOSS VÀ VAL LOSS
    # ======================================
    epoch_train_loss = r_train_loss / len(train_loader.dataset)
    epoch_val_loss = r_val_loss / len(val_loader.dataset)

    train_losses.append(epoch_train_loss)
    val_losses.append(epoch_val_loss)

    # ======================================
    # 4. CẬP NHẬT LEARNING RATE
    # ======================================
    scheduler.step(epoch_val_loss)

    # ======================================
    # 5. LƯU MÔ HÌNH TỐT NHẤT
    # ======================================
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss

        torch.save(
            model.state_dict(),
            'best_ccpp_model.pth'
        )

    # In kết quả mỗi 20 Epoch
    if (epoch + 1) % 20 == 0:
        current_lr = optimizer.param_groups[0]['lr']

        print(
            f"Epoch {epoch+1:3d}: "
            f"Train Loss: {epoch_train_loss:.6f} | "
            f"Val Loss: {epoch_val_loss:.6f} | "
            f"LR: {current_lr:.6f}"
        )

# ==========================================
# 6. TRỰC QUAN HÓA TRAINING - VALIDATION LOSS
# ==========================================

plt.figure(figsize=(10, 5))

plt.plot(
    train_losses,
    label='Training Loss (Huber)',
    color='teal'
)

plt.plot(
    val_losses,
    label='Validation Loss (Huber)',
    color='magenta'
)

plt.title('Training and Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nBest Validation Loss: {best_val_loss:.6f}")

In [ ]:
# ==========================================
# 4. ĐÁNH GIÁ VÀ TRIỂN KHAI MÔ HÌNH (TESTING)
# ==========================================

# Tải mô hình tốt nhất đã lưu
model.load_state_dict(torch.load('best_ccpp_model.pth'))
model.eval()

with torch.no_grad():
    X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32).to(device)
    y_pred_scaled = model(X_test_tensor).cpu().numpy()
    
# Nghịch đảo chuẩn hóa để về đơn vị MW
y_pred = scaler_y.inverse_transform(y_pred_scaled)
y_actual = y_test

# Các chỉ số định lượng
mae = mean_absolute_error(y_actual, y_pred)
mse = mean_squared_error(y_actual, y_pred)
r2 = r2_score(y_actual, y_pred)

print(f"\n--- CHỈ SỐ ĐÁNH GIÁ TRÊN TẬP TEST ---")
print(f"MAE: {mae:.2f} MW")
print(f"MSE: {mse:.2f}")
print(f"R² Score: {r2:.4f}")

# Trực quan hóa kết quả dự báo
fig, ax = plt.subplots(1, 2, figsize=(16, 6))

# 1. So sánh Thực tế vs Dự báo (100 mẫu)
ax[0].plot(y_actual[:100], label='Thực tế', color='dodgerblue', marker='o', markersize=4)
ax[0].plot(y_pred[:100], label='Dự báo', color='orangered', linestyle='--', marker='x', markersize=4)
ax[0].set_title('So sánh Thực tế vs Dự báo (100 mẫu)')
ax[0].legend()

# 2. Biểu đồ Regression (Scatter plot)
ax[1].scatter(y_actual, y_pred, alpha=0.3, s=10, color='forestgreen')
ax[1].plot([y_actual.min(), y_actual.max()], [y_actual.min(), y_actual.max()], 'r--', lw=2)
ax[1].set_xlabel('Giá trị thực tế (MW)')
ax[1].set_ylabel('Giá trị dự báo (MW)')
ax[1].set_title(f'Tương quan Dự báo (R² = {r2:.4f})')

plt.tight_layout()
plt.show()

# --- TRIỂN KHAI (Dự báo cho 1 mẫu mới) ---
def predict_new(at, v, ap, rh):
    model.eval()
    sample = np.array([[at, v, ap, rh]])
    sample_scaled = scaler_X.transform(sample)
    sample_tensor = torch.tensor(sample_scaled, dtype=torch.float32).to(device)
    
    with torch.no_grad():
        pred_scaled = model(sample_tensor).cpu().numpy()
        pred = scaler_y.inverse_transform(pred_scaled)
    return pred[0][0]

# Giả sử mẫu mới: AT=23.64, V=58.49, AP=1011.4, RH=74.2
print(f"\nDự báo cho mẫu mới: {predict_new(23.64, 58.49, 1011.4, 74.2):.2f} MW")


In [ ]:
# ==========================================
# 5. ĐỊNH LƯỢNG SAI SỐ VÀ ĐÁNH GIÁ HỆ THỐNG
# ==========================================
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Đảm bảo mô hình ở chế độ đánh giá
model.eval()

with torch.no_grad():
    # 1. Dự báo trên toàn bộ tập kiểm tra (test_loader)
    all_preds = []
    all_actuals = []
    
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        outputs = model(inputs)
        
        all_preds.append(outputs.cpu().numpy())
        all_actuals.append(labels.numpy())

    # Chuyển list thành mảng numpy
    y_pred_scaled = np.vstack(all_preds)
    y_true_scaled = np.vstack(all_actuals)

    # 2. Nghịch đảo chuẩn hóa để đưa về đơn vị MW (Megawatts)
    y_pred = scaler_y.inverse_transform(y_pred_scaled)
    y_true = scaler_y.inverse_transform(y_true_scaled)

# 3. Tính toán các chỉ số định lượng
mae = mean_absolute_error(y_true, y_pred)
mse = mean_squared_error(y_true, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_true, y_pred)

# 4. Hiển thị kết quả
print("-" * 30)
print(f"{'CHỈ SỐ ĐÁNH GIÁ':^30}")
print("-" * 30)
print(f"Mean Absolute Error (MAE):      {mae:.4f} MW")
print(f"Mean Squared Error (MSE):       {mse:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.4f} MW")
print(f"R-squared Score (R²):           {r2:.4f}")
print("-" * 30)

# 5. Giải thích ý nghĩa ngắn gọn
print(f"\nÝ nghĩa: Trung bình mô hình dự báo lệch khoảng {mae:.2f} MW so với sản lượng thực tế.")
if r2 > 0.9:
    print("Đánh giá: Mô hình có độ chính xác rất cao (R² > 0.9).")